### ProShares Short VIX Short-Term Futures ETF

In [3]:
import os
import datetime as dt

from dotenv import load_dotenv
from pathlib import Path

import json
import time
from urllib.parse import urljoin, urlencode
import requests

import numpy as np
import pandas as pd

import plotly
import plotly.graph_objs as go
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt

from scipy.stats import spearmanr
from statsmodels.graphics.tsaplots import plot_acf
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    roc_auc_score
)

from Modules.utility import Utility
from Modules.show_plot import ShowPlot
from Modules.request_api_v2 import RequestApi
from Modules.get_market_data import GetMarketData
from Modules.stock_prices_and_market_data import ClassStockPricesMarketData
from Modules.financial import Financial
from Modules.pdf_url_to_markdown import PDFUrlToMarkdown
from Modules.webpage_to_markdown import WebpageToMarkdown
from Modules.compute_fibonacci_extension import ComputeFibonacciExtension
from Modules.fibonacci_retracement import FibonacciRetracement
from Modules.fibonacci_extension import FibonacciExtension

In [4]:
# FRED API key for accessing Federal Reserve Economic Data
FRED_API_KEY = os.getenv("FRED_API_KEY")
# Output directory for storing downloaded data
API_BASE_URL = os.getenv("API_BASE_URL", "http://localhost:8000")
request_api = RequestApi(API_BASE_URL)
fred_url = "https://api.stlouisfed.org/fred/series/observations"
get_market_data = GetMarketData(Path('/workspace/data'))
utility = Utility()
stock_prices_market_data = ClassStockPricesMarketData()
financial = Financial()
# URLからPDFをダウンロードしてMarkdownに変換するクラスのインスタンスを作成
pdf_to_md = PDFUrlToMarkdown()
# WEBページをMarkdownに変換する関数
webpage_to_markdown = WebpageToMarkdown()
# Fibonacciエクステンションを計算するクラスのインスタンスを作成
fibonacci_extension = ComputeFibonacciExtension()
# Fibonacciエクステンションをプロットするクラスのインスタンスを作成
fibonacci_extension_v2 = FibonacciExtension() 
# Fibonacciリトレースメントを計算するクラスのインスタンスを作成
fibonacci_retracement = FibonacciRetracement()

In [6]:
# Date range for fetching API s
start = '2010-01-01'
end = dt.datetime.now().strftime('%Y-%m-%d')
output_dir = Path("/workspace/data/")
output_dir.mkdir(parents=True, exist_ok=True)
frequency = "d"
aggregation_method = "avg"

#### S&P500の時系列データを取得

In [7]:
sp500_df = request_api.fetch_stock_timeseries_data(
    start_date=start,
    end_date=end,
    code="^GSPC",
    market=None,
)
sp500_df.tail()

取得件数: 4442


,id,stock_code,stock_market,date,open,high,low,close,volume,ma5,...,lower1,rci52,cross,gc,dc,macd_gc,macd_dc,rci_gc,rci_dc,rising_condition
4437,1372719,^GSPC,None,2026-10-27,7722.600098,7736.240234,7677.160156,7699.470215,5160120000,7681.480176,...,7674.871703,18.552036,False,NaN,NaN,NaN,NaN,NaN,NaN,False
4438,1374591,^GSPC,None,2026-10-28,7743.979980,7758.430176,7718.370117,7735.560059,5466960000,7681.306152,...,7674.310572,27.456672,False,NaN,NaN,NaN,NaN,1.666667,NaN,False
4439,1374599,^GSPC,None,2026-10-29,7720.390137,7732.209961,7690.939941,7699.609863,5827600000,7682.304102,...,7676.715761,33.040212,False,NaN,NaN,NaN,NaN,NaN,NaN,False
4440,1374607,^GSPC,None,2026-10-30,7580.069824,7637.479980,7527.560059,7631.069824,6149170000,7681.082031,...,7670.226174,28.634850,False,NaN,NaN,NaN,NaN,NaN,-39.555556,False
4441,1376914,^GSPC,None,2026-11-02,7541.399902,7662.229980,7539.379883,7610.830078,3558585000,7675.308008,...,7661.393479,24.494152,False,NaN,NaN,NaN,NaN,NaN,NaN,False


#### Nasdaq 100の時系列データを取得

In [8]:
qqq_df = request_api.fetch_stock_timeseries_data(
    start_date=start,
    end_date=end,
    code="^IXIC",
    market=None,
)
qqq_df.tail()

取得件数: 4442


,id,stock_code,stock_market,date,open,high,low,close,volume,ma5,...,lower1,rci52,cross,gc,dc,macd_gc,macd_dc,rci_gc,rci_dc,rising_condition
4437,1372721,^IXIC,None,2026-10-27,26477.810547,26540.810547,26219.310547,26286.310547,7914230000,26223.453906,...,26389.878450,31.102194,False,NaN,NaN,NaN,NaN,-50.0,NaN,False
4438,1374595,^IXIC,None,2026-10-28,26549.849609,26587.339844,26414.919922,26541.609375,7985070000,26211.429688,...,26365.216353,34.022027,False,NaN,NaN,NaN,NaN,NaN,NaN,True
4439,1374603,^IXIC,None,2026-10-29,26184.720703,26286.460938,26053.160156,26201.820312,7405880000,26190.111719,...,26328.123979,32.920686,False,NaN,NaN,NaN,NaN,NaN,NaN,False
4440,1374609,^IXIC,None,2026-10-30,25437.000000,25673.000000,25177.330078,25592.490234,9068930000,26129.819922,...,26218.753653,25.382054,False,NaN,NaN,NaN,NaN,NaN,NaN,False
4441,1376916,^IXIC,None,2026-11-02,25164.363281,25799.455078,25145.951172,25600.173828,8002403000,26044.480859,...,26121.367753,18.748399,False,NaN,NaN,NaN,NaN,NaN,NaN,False


#### SVXY時系列を取得

In [9]:
svxy_df = request_api.fetch_stock_timeseries_data(
    start_date=start,
    end_date=end,
    code="SVXY",
    market=None,
)
svxy_df.tail()

取得件数: 3792


,id,stock_code,stock_market,date,open,high,low,close,volume,ma5,...,lower1,rci52,cross,gc,dc,macd_gc,macd_dc,rci_gc,rci_dc,rising_condition
3787,1374655,SVXY,None,2026-10-27,62.610001,62.799999,62.259998,62.540001,728500,62.606001,...,62.933147,61.307948,False,NaN,NaN,NaN,NaN,NaN,NaN,False
3788,1374656,SVXY,None,2026-10-28,63.270000,63.400002,62.529999,62.639999,1399400,62.348000,...,62.832362,55.493896,False,NaN,NaN,NaN,NaN,NaN,NaN,False
3789,1374657,SVXY,None,2026-10-29,63.930000,63.930000,63.020000,63.119999,1338400,62.322000,...,62.800253,50.755571,False,NaN,NaN,NaN,NaN,NaN,NaN,False
3790,1374658,SVXY,None,2026-10-30,60.549999,62.209999,58.299999,62.099998,3982200,62.512000,...,62.673429,42.516862,False,NaN,NaN,NaN,NaN,NaN,NaN,False
3791,1376918,SVXY,None,2026-11-02,60.520000,62.450001,60.369999,61.369999,2652400,62.353999,...,62.476196,31.631521,False,NaN,NaN,NaN,NaN,NaN,NaN,False


#### チャート

In [10]:
def stock_prices_and_index(
        code: str,
        name: str,
        start: str,
        end: str,
        market: str | None = None,
        df_sp: pd.DataFrame | None = None,
        index_name: str | None = None,
        df_index: pd.DataFrame | None = None,
    ):
    # /api/v1/time_series_data/stock/
    response = request_api.get_stock_time_series_data(
        code=code,
        market=market,
        start=start,
        end=end
    )
    df_stock = pd.DataFrame(response)

    # S&P500を統合
    if df_sp is not None:
        df_sp_tmp = df_sp.copy() if df_sp is not None else pd.DataFrame()
        if "date" not in df_sp_tmp.columns:
            df_sp_tmp = df_sp_tmp.reset_index()

        df_sp_tmp["date"] = pd.to_datetime(df_sp_tmp["date"])
        df_sp_tmp = df_sp_tmp.set_index("date")
        df_sp_tmp = df_sp_tmp.loc[start:end]

    # INDEXを統合
    if df_index is not None:
        df_index_tmp = df_index.copy() if df_index is not None else pd.DataFrame()
        if "date" not in df_index_tmp.columns:
            df_index_tmp = df_index_tmp.reset_index()
        df_index_tmp["date"] = pd.to_datetime(df_index_tmp["date"])
        df_index_tmp = df_index_tmp.set_index("date")
        df_index_tmp = df_index_tmp.loc[start:end]

    # 株価時系列データを整形
    df = df_stock.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.set_index("date")
    df = df.loc[start:end]


    # インデックスを揃えて結合
    if df_sp is not None:
        df["MA50_SP"] = df_sp_tmp["ma50"].reindex(df.index)
        df["MA200_SP"] = df_sp_tmp["ma200"].reindex(df.index)
    if df_index is not None:
        df["MA50_" + index_name] = df_index_tmp["ma50"].reindex(df.index)
        df["MA200_" + index_name] = df_index_tmp["ma200"].reindex(df.index)

    show_plot = ShowPlot()
    fig = show_plot.create_basic_chart(
        df=df.reset_index(),
        code=code,
        name=name,
        start=start,
        end=end,
        ma5=False,
        ma25=False,
        ma50=True,
        ma100=False,
        ma150=False,
        ma200=True,
    )
    # ★ 2つのY軸を定義（左：HYMC、右：SP500 & GOLD）
    fig.update_layout(
        yaxis=dict(
            title=f"{name} Price",
            side="left"
        ),
        yaxis2=dict(
            title="SP500 / WTI",
            overlaying="y",
            side="right"
        )
    )

    # --- SP500（右軸） ---
    if df_sp is not None:
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA50_SP"],
                name="SP_MA50",
                line={"color": "blue", "width": 1.2},
                yaxis="y2"   # ★ 右軸
            )
        )
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA200_SP"],
                name="SP_MA200",
                line={"color": "gray", "width": 1.2},
                yaxis="y2"   # ★ 右軸
            )
        )

    # --- INDEX（右軸） ---
    if df_index is not None:
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA50_" + index_name],
                name=index_name + "_MA50",
                line={"color": "orange", "width": 1.2},
                yaxis="y2"   # ★ 右軸
            )
        )
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA200_" + index_name],
                name=index_name + "_MA200",
                line={"color": "yellow", "width": 1.2},
                yaxis="y2"   # ★ 右軸
            )
        )

    return fig

#### SVXYのチャート

In [11]:
name = "ProShares Short QQQ"
start = dt.datetime(2025, 1, 1).strftime("%Y-%m-%d")
# グラフ領域の作成
fig = stock_prices_and_index(
    code="SVXY",
    name=name,
    start=start,
    end=end,
    market=None,
    df_sp=None,
    df_index=None
)
fig.show()

取得件数: 667


In [12]:
# 例: すでに作成済みの fig を保存
output_dir = Path("/workspace/data/figures")
output_dir.mkdir(parents=True, exist_ok=True)
# ファイル名にコードと期間を含める
output_path = output_dir / f"SVXY_{start}_{end}.png"
# 既存の画像ファイルが存在する場合は削除
if output_path.exists():
    output_path.unlink()
# 画像ファイルを保存
fig.write_image(str(output_path), width=1600, height=900, scale=2)
# 保存完了のメッセージを表示
print("saved:", output_path)

saved: /workspace/data/figures/SVXY_2025-01-01_2026-11-03.png


#### 時系列データを保存

In [14]:
output_dir = Path("/workspace/data/")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"SVXY_data_{dt.datetime.now().strftime('%Y-%m-%d')}.csv"
# 既存の CSV ファイルが存在する場合は削除
if output_path.exists():
    output_path.unlink()
# CSV ファイルを新規作成する準備完了
df = svxy_df.copy()
df = svxy_df.reset_index()
df["date"] = pd.to_datetime(df["date"])
df = df.set_index("date")
df = df.loc['2026-01-01':end]
df.to_csv(output_path)
df.tail()

,index,id,stock_code,stock_market,open,high,low,close,volume,ma5,...,lower1,rci52,cross,gc,dc,macd_gc,macd_dc,rci_gc,rci_dc,rising_condition
date,,,,,,,,,,,,,,,,,,,,,
2026-10-27,3787,1374655,SVXY,None,62.610001,62.799999,62.259998,62.540001,728500,62.606001,...,62.933147,61.307948,False,NaN,NaN,NaN,NaN,NaN,NaN,False
2026-10-28,3788,1374656,SVXY,None,63.270000,63.400002,62.529999,62.639999,1399400,62.348000,...,62.832362,55.493896,False,NaN,NaN,NaN,NaN,NaN,NaN,False
2026-10-29,3789,1374657,SVXY,None,63.930000,63.930000,63.020000,63.119999,1338400,62.322000,...,62.800253,50.755571,False,NaN,NaN,NaN,NaN,NaN,NaN,False
2026-10-30,3790,1374658,SVXY,None,60.549999,62.209999,58.299999,62.099998,3982200,62.512000,...,62.673429,42.516862,False,NaN,NaN,NaN,NaN,NaN,NaN,False
2026-11-02,3791,1376918,SVXY,None,60.520000,62.450001,60.369999,61.369999,2652400,62.353999,...,62.476196,31.631521,False,NaN,NaN,NaN,NaN,NaN,NaN,False
